# L5–L6 Kernel Methods

## Regularization, nonlinear features, SVMs, RKHS, kernel regression, and local averaging

This notebook integrates the original L5 notes with the L6 material and the relevant additions from:

- L5.pdf and L6.pdf;
- Francis Bach, Learning Theory from First Principles, especially Chapters 3, 4, 6, 7, and 12;
- Understanding Machine Learning: Solution Manual, especially Chapters 13, 15, 16, and 22.

Every numbered section is an independent Markdown cell. We use the averaged empirical-risk convention unless explicitly stated:

$$
\widehat R_n(f)=\frac1n\sum_{i=1}^n \ell(y_i,f(x_i)).
$$

Changing whether the loss or regularizer contains a factor \(1/2\) only rescales \(\lambda\); it does not change the estimator's structure.

## Section 1 — Linear prediction, least squares, and design regimes

A model is linear when it is linear in its parameter:

$$
f_\theta(x)=\phi(x)^\top\theta.
$$

It need not be linear in the original input \(x\). The feature map \(\phi(x)\) may contain polynomials, trigonometric functions, text features, or infinitely many coordinates.

For a design matrix \(\Phi\in\mathbb R^{n\times d}\) and labels \(y\in\mathbb R^n\), ordinary least squares solves

$$
\min_\theta \frac1n\|y-\Phi\theta\|_2^2.
$$

If \(\Phi\) has full column rank,

$$
\widehat\theta_{\mathrm{OLS}}
=(\Phi^\top\Phi)^{-1}\Phi^\top y,
\qquad
\widehat y=P_\Phi y,
$$

where

$$
P_\Phi=\Phi(\Phi^\top\Phi)^{-1}\Phi^\top
$$

is the orthogonal projector onto \(\operatorname{im}(\Phi)\). Hence

$$
\Phi^\top(y-\widehat y)=0.
$$

The explicit inverse is mainly an analysis device. Numerically, QR factorization, Cholesky factorization, conjugate gradients, or gradient methods are preferable.

Two risk settings must be separated:

- Fixed design: the inputs are fixed and only responses are resampled. This is an in-sample denoising problem.
- Random design: \(X\sim p_X\), and risk is evaluated on a fresh input. This is the usual generalization problem.

Under a correctly specified, full-rank fixed-design model with noise variance \(\sigma^2\),

$$
\mathbb E[R(\widehat\theta_{\mathrm{OLS}})]-R^*
=\frac{\sigma^2d}{n}.
$$

For a fresh response at the same design points,

$$
\mathbb E[\mathrm{MSE}_{\mathrm{train}}]
=\sigma^2\left(1-\frac dn\right),
\qquad
\mathbb E[\mathrm{MSE}_{\mathrm{test}}]
=\sigma^2\left(1+\frac dn\right).
$$

Thus training error is optimistically biased by \(2\sigma^2d/n\).

## Section 2 — Ridge regression: objective, primal form, and dual form

Ridge regression adds an \(\ell_2\) penalty:

$$
\widehat\theta_\lambda
\in\arg\min_\theta
\left\{
\frac1n\|y-\Phi\theta\|_2^2
+\lambda\|\theta\|_2^2
\right\}.
$$

The normal equation is

$$
(\Phi^\top\Phi+n\lambda I_d)\widehat\theta_\lambda
=\Phi^\top y,
$$

so

$$
\boxed{
\widehat\theta_\lambda
=(\Phi^\top\Phi+n\lambda I_d)^{-1}\Phi^\top y.
}
$$

Using the matrix inversion identity,

$$
\boxed{
\widehat\theta_\lambda
=\Phi^\top(\Phi\Phi^\top+n\lambda I_n)^{-1}y.
}
$$

The two forms determine the computational strategy:

- if \(d\ll n\), solve the \(d\times d\) primal system;
- if \(n\ll d\), solve the \(n\times n\) dual system;
- in an infinite feature space, the dual form remains meaningful if inner products can be evaluated.

Adding \(\lambda I\) shifts every covariance eigenvalue:

$$
\mu_j(\widehat\Sigma+\lambda I)=\mu_j(\widehat\Sigma)+\lambda,
\qquad
\widehat\Sigma=\frac1n\Phi^\top\Phi.
$$

Therefore ridge improves conditioning and prevents poorly observed directions from acquiring extremely large coefficients.

For Gaussian noise and the Gaussian prior

$$
\theta\sim
\mathcal N\left(0,\frac{\sigma^2}{n\lambda}I\right),
$$

the ridge estimator is both the posterior mean and the MAP estimator.

## Section 3 — Bias–variance and approximation–estimation decompositions

For a random predictor \(\widehat f\), a target \(f_*\), and zero-mean independent noise \(\varepsilon\),

$$
Y=f_*(X)+\varepsilon,
$$

the squared prediction error decomposes as

$$
\boxed{
\mathbb E[(Y-\widehat f(X))^2]
=
\sigma^2
+
\left(f_*(X)-\mathbb E[\widehat f(X)]\right)^2
+
\operatorname{Var}(\widehat f(X)).
}
$$

The three terms are irreducible noise, squared bias, and variance.

This differs from the learning-theory decomposition. If

$$
f_{\mathcal F}^*\in\arg\min_{f\in\mathcal F}R(f),
$$

then

$$
R(\widehat f)-R^*
=
\underbrace{R(f_{\mathcal F}^*)-R^*}_{\text{approximation error}}
+
\underbrace{R(\widehat f)-R(f_{\mathcal F}^*)}_{\text{estimation error}}.
$$

Increasing model capacity tends to reduce approximation error but increase estimation error. Increasing regularization tends to increase bias or approximation error but reduce variance or estimation error.

The Gauss–Markov theorem states that OLS is the best linear unbiased estimator under the classical assumptions. The word unbiased is essential: ridge may have smaller prediction risk because a small increase in bias can be outweighed by a large decrease in variance.

## Section 4 — Ridge as spectral shrinkage and effective degrees of freedom

Let

$$
\widehat\Sigma
=U\operatorname{diag}(\mu_1,\ldots,\mu_d)U^\top,
\qquad
a_j=u_j^\top\theta_*.
$$

Under the correctly specified fixed-design model,

$$
\begin{aligned}
\mathbb E[R(\widehat\theta_\lambda)]-R^*
={}&
\lambda^2\theta_*^\top
(\widehat\Sigma+\lambda I)^{-2}
\widehat\Sigma\theta_*\\
&+
\frac{\sigma^2}{n}
\operatorname{tr}\left[
\widehat\Sigma^2(\widehat\Sigma+\lambda I)^{-2}
\right].
\end{aligned}
$$

Equivalently,

$$
\operatorname{Bias}^2
=
\sum_j
\frac{\lambda^2\mu_j}{(\mu_j+\lambda)^2}a_j^2,
$$

$$
\operatorname{Variance}
=
\frac{\sigma^2}{n}
\sum_j
\left(\frac{\mu_j}{\mu_j+\lambda}\right)^2.
$$

The shrinkage multiplier is

$$
s_j(\lambda)=\frac{\mu_j}{\mu_j+\lambda}.
$$

Directions with \(\mu_j\gg\lambda\) are mostly retained; directions with \(\mu_j\ll\lambda\) are strongly suppressed.

Two related notions of effective dimension occur:

$$
df_1(\lambda)=\sum_j\frac{\mu_j}{\mu_j+\lambda}
=\operatorname{tr}(H_\lambda),
$$

$$
df_2(\lambda)=\sum_j
\left(\frac{\mu_j}{\mu_j+\lambda}\right)^2
=\operatorname{tr}(H_\lambda^2).
$$

\(df_1\) is a soft parameter count; \(df_2\) directly controls fixed-design variance.

PCA followed by OLS performs hard spectral truncation. Ridge performs soft spectral shrinkage.

## Section 5 — Why regularization generalizes

Regularization helps through four connected mechanisms:

1. Conditioning: it removes nearly flat directions from the optimization problem.
2. Capacity control: it restricts the norm of the learned predictor.
3. Stability: replacing one training example changes the solution only slightly.
4. Prior structure: the regularizer encodes which solutions are considered simple.

For a \(G\)-Lipschitz loss, features satisfying \(\|\phi(x)\|\le R\), and an \(\ell_2\)-regularized objective, the expected stability term is typically

$$
O\left(\frac{G^2R^2}{\lambda n}\right).
$$

If an algorithm is \(\epsilon_{\mathrm{stab}}\)-stable and is an \(\epsilon_{\mathrm{ERM}}\)-approximate empirical-risk minimizer, then

$$
\boxed{
\mathbb E[R(A(S))-R(f^*)]
\le
\epsilon_{\mathrm{stab}}+\epsilon_{\mathrm{ERM}}.
}
$$

Thus larger \(\lambda\) improves stability but introduces more regularization bias.

For a finite collection of \(M\) candidate models or hyperparameters, model selection generally costs a term of order

$$
\sqrt{\frac{\log M}{n}}.
$$

Validation and structural risk minimization make this complexity cost explicit. A validation split should not be so small that the \(\sqrt{\log M/n_{\mathrm{val}}}\) term dominates.

## Section 6 — Regularization beyond ridge

The general regularized ERM problem is

$$
\min_\theta
\left\{
\widehat R_n(\theta)+\lambda\Omega(\theta)
\right\}.
$$

Common choices are:

- \(\Omega(\theta)=\|\theta\|_2^2\): small, dense coefficients;
- \(\Omega(\theta)=\|\theta\|_1\): sparse coefficients;
- \(\Omega(\theta)=\theta^\top M\theta\), \(M\succ0\): direction-dependent Tikhonov regularization;
- a constraint such as \(\|\theta\|\le D\): constrained capacity control.

For the scalar problem

$$
\min_w\left\{\frac12w^2-xw+\lambda|w|\right\},
$$

the solution is the soft-threshold operator

$$
\boxed{
w=\operatorname{sign}(x)(|x|-\lambda)_+.
}
$$

This explains why \(\ell_1\) regularization can set coefficients exactly to zero, whereas \(\ell_2\) regularization usually only shrinks them.

Regularization may also be implicit. Initialization, gradient descent, early stopping, architecture, data augmentation, and stochasticity can select one solution among many empirical minimizers.

## Section 7 — Nonlinear feature maps

Nonlinear prediction can be obtained by applying a linear method to nonlinear features:

$$
h(x)=\operatorname{sign}(w^\top\phi(x)+b).
$$

For two variables, the quadratic map

$$
\phi(x_1,x_2)
=
(x_1,x_2,x_1^2,x_1x_2,x_2^2)
$$

turns a hyperplane in feature space into a quadratic boundary in input space.

For \(d\) variables, the number of monomials of degree exactly \(s\) is

$$
\binom{d+s-1}{s}.
$$

The number of monomials of degree at most \(s\), including the constant, is

$$
\binom{d+s}{s}.
$$

Any finite hypothesis class can be linearized. For

$$
\mathcal H=\{h_1,\ldots,h_M\},
$$

define a feature indicating the output of every hypothesis and append a constant coordinate. Then each \(h_j\) is represented by a linear vector such as \((2e_j,-1)\). The difficulty is not existence but potentially enormous feature dimension.

Feature design is an inductive bias. Handcrafted features, implicit kernel features, and learned neural features are three different ways to choose that bias.

## Section 8 — SVM geometry and margins

For a hyperplane

$$
w^\top x+b=0,
$$

the Euclidean distance of \(x\) to the hyperplane is

$$
\frac{|w^\top x+b|}{\|w\|}.
$$

On linearly separable data, maximizing the minimum geometric margin is equivalent to the hard-margin SVM:

$$
\min_{w,b}\frac12\|w\|^2
\quad
\text{subject to}
\quad
y_i(w^\top x_i+b)\ge1.
$$

The normalization by \(1\) is possible because multiplying \(w,b\) by a positive constant does not change the classifier.

If \(\|x_i\|\le\rho\) and a unit vector separates the data with margin \(\gamma\), the perceptron mistake bound is

$$
\boxed{
T_{\mathrm{mistakes}}
\le
\left(\frac{\rho}{\gamma}\right)^2.
}
$$

Thus margin is both a geometric complexity measure and an algorithmic convergence quantity.

## Section 9 — Soft-margin SVM, hinge loss, duality, and support vectors

The soft-margin primal is

$$
\min_{w,b,\xi}
\frac12\|w\|^2+C\sum_i\xi_i
$$

subject to

$$
y_i(w^\top x_i+b)\ge1-\xi_i,
\qquad
\xi_i\ge0.
$$

After eliminating the slack variables,

$$
\min_{w,b}
\frac1n\sum_i(1-y_i(w^\top x_i+b))_+
+\frac\lambda2\|w\|^2,
$$

with

$$
\boxed{\lambda=\frac1{nC}}.
$$

The kernelized dual is

$$
\max_\alpha
\sum_i\alpha_i
-\frac12\sum_{i,j}
\alpha_i\alpha_jy_iy_jK(x_i,x_j)
$$

subject to

$$
0\le\alpha_i\le C,
\qquad
\sum_i\alpha_i y_i=0.
$$

At optimum,

$$
w=\sum_i\alpha_i y_i\phi(x_i).
$$

KKT conditions give:

- margin \(>1\): \(\alpha_i=0\);
- margin \(=1\): usually \(0<\alpha_i<C\);
- inside the margin or misclassified: often \(\alpha_i=C\).

If \(\|x\|\le R\), hinge loss is \(R\)-Lipschitz in \(w\):

$$
|\ell(w_1)-\ell(w_2)|
\le R\|w_1-w_2\|.
$$

Even when data are separable, finite-\(C\) soft SVM need not equal hard SVM: it may be cheaper to violate a difficult point than to use an extremely large norm.

## Section 10 — The kernel trick and kernelized linear algorithms

Suppose a feature map \(\phi:\mathcal X\to\mathcal H\) satisfies

$$
k(x,z)=\langle\phi(x),\phi(z)\rangle_{\mathcal H}.
$$

Any algorithm that uses feature vectors only through inner products can replace those inner products with kernel evaluations.

For SVM,

$$
h(x)=
\operatorname{sign}
\left(
\sum_i\alpha_i y_i k(x_i,x)+b
\right).
$$

The feature vectors need never be constructed.

The kernelized perceptron uses a signed coefficient convention:

$$
w^{(t)}=\sum_i\alpha_i^{(t)}\phi(x_i).
$$

On a mistake at example \(i\),

$$
y_i\sum_j\alpha_jK(x_j,x_i)\le0,
\qquad
\alpha_i\leftarrow\alpha_i+y_i.
$$

Prediction is

$$
\operatorname{sign}\left(\sum_i\alpha_iK(x_i,x)\right).
$$

An equivalent convention uses nonnegative mistake counts \(c_i\) and writes \(w=\sum_i c_i y_i\phi(x_i)\). The two conventions must not be mixed.

Nearest-centroid classification, PCA, and many other dot-product algorithms can also be kernelized.

## Section 11 — Valid kernels and Gram matrices

A real-valued symmetric function \(k\) is a valid positive-semidefinite kernel if, for every finite set \(x_1,\ldots,x_n\), the Gram matrix

$$
K_{ij}=k(x_i,x_j)
$$

satisfies

$$
c^\top Kc\ge0
\quad
\text{for every }c\in\mathbb R^n.
$$

Equivalent statements are:

1. every Gram matrix is positive semidefinite;
2. there exists a Hilbert-space feature map with
   \(k(x,z)=\langle\phi(x),\phi(z)\rangle\).

Useful closure rules include:

- if \(k_1,k_2\) are kernels and \(a,b\ge0\), then \(ak_1+bk_2\) is a kernel;
- \(k_1k_2\) is a kernel;
- if \(g\) is any function, \(g(x)k(x,z)g(z)\) is a kernel;
- pointwise limits of kernels remain kernels when the limit exists.

Two instructive constructions are

$$
k(i,j)=\min(i,j)
$$

using prefix-indicator features, and

$$
k(E,E')=|E\cap E'|
$$

using set-indicator features.

A merely nonnegative function is not necessarily a valid kernel. Positive semidefiniteness concerns quadratic forms of entire Gram matrices.

## Section 12 — Important kernel families and polynomial feature dimensions

The linear kernel is

$$
k(x,z)=x^\top z.
$$

The homogeneous polynomial kernel is

$$
k_s(x,z)=(x^\top z)^s.
$$

Using multi-indices \(\alpha\in\mathbb N^d\), \(|\alpha|=s\), one normalized feature is

$$
\phi_\alpha(x)
=
\sqrt{\frac{s!}{\alpha_1!\cdots\alpha_d!}}x^\alpha.
$$

Its minimal explicit dimension is

$$
\binom{d+s-1}{s}.
$$

The inhomogeneous kernel

$$
k_s(x,z)=(c+x^\top z)^s,
\qquad c\ge0,
$$

contains all degrees up to \(s\), with dimension \(\binom{d+s}{s}\) when \(c>0\).

The Gaussian RBF kernel is

$$
k(x,z)
=
\exp\left(-\frac{\|x-z\|^2}{2\sigma^2}\right)
=\exp(-\gamma\|x-z\|^2).
$$

The Laplacian kernel is

$$
k(x,z)=\exp\left(-\frac{\|x-z\|}{r}\right).
$$

Matérn kernels form a family whose parameter controls finite smoothness. Kernels can also be designed for sets, bags of words, sequences, strings, trees, graphs, point clouds, and probability distributions.

## Section 13 — Translation-invariant kernels and the Fourier viewpoint

Let

$$
k(x,z)=q(x-z).
$$

Bochner's theorem states that a continuous translation-invariant function is a positive-semidefinite kernel exactly when it is the Fourier transform of a finite nonnegative measure. When a spectral density exists, it is nonnegative:

$$
\widehat q(\omega)\ge0.
$$

The associated RKHS norm has the spectral form

$$
\boxed{
\|f\|_{\mathcal H}^2
=
\frac1{(2\pi)^d}
\int_{\mathbb R^d}
\frac{|\widehat f(\omega)|^2}{\widehat q(\omega)}
\,d\omega.
}
$$

Thus the kernel specifies the cost of every frequency:

- large \(\widehat q(\omega)\): the frequency is inexpensive;
- small \(\widehat q(\omega)\): the frequency is strongly penalized.

Multiplication by \(\omega_j\) in the Fourier domain corresponds to differentiation in the input domain. Therefore spectral decay determines smoothness:

- Laplacian or exponential kernels correspond to finite-order Sobolev control;
- Matérn kernels allow an explicit finite smoothness order;
- Gaussian kernels penalize high frequencies extremely strongly and produce a smaller, very smooth RKHS.

A common bandwidth heuristic is to choose \(r\) from a quantile, often the median, of pairwise training distances. It is only a heuristic and should normally be checked by validation.

## Section 14 — RKHS construction and the reproducing property

Starting from a valid kernel, consider finite expansions

$$
\mathcal H_0
=
\left\{
\sum_i\alpha_i k(\cdot,x_i)
\right\}.
$$

Define

$$
\left\langle
\sum_i\alpha_i k(\cdot,x_i),
\sum_j\beta_j k(\cdot,z_j)
\right\rangle_{\mathcal H}
=
\sum_{i,j}\alpha_i\beta_jk(x_i,z_j).
$$

After taking the Hilbert-space completion, one obtains the canonical reproducing-kernel Hilbert space.

Its defining identity is

$$
\boxed{
\langle k(\cdot,x),f\rangle_{\mathcal H}=f(x).
}
$$

Consequently,

$$
|f(x)|
\le
\|f\|_{\mathcal H}\sqrt{k(x,x)}.
$$

If \(\sup_x k(x,x)\le R^2\), then

$$
\|f\|_\infty\le R\|f\|_{\mathcal H}.
$$

Not every Hilbert space of functions is an RKHS. In \(L^2\), point evaluation is not continuous and functions are defined only up to equality almost everywhere.

Feature maps are not unique: coordinates may be rotated, duplicated, or embedded in a larger space. The canonical RKHS generated by the kernel is nevertheless unique up to isometric isomorphism.

## Section 15 — Representer theorem and minimum-norm interpolation

Consider an objective that accesses \(f\) only through training-point evaluations and its RKHS norm:

$$
\Psi\left(
f(x_1),\ldots,f(x_n),\|f\|_{\mathcal H}^2
\right).
$$

If \(\Psi\) is strictly increasing in the norm argument, every minimizer lies in the span of the kernel sections:

$$
\boxed{
f(\cdot)=\sum_{i=1}^n\alpha_i k(\cdot,x_i).
}
$$

The proof decomposes

$$
f=f_D+f_\perp,
$$

where \(f_D\) is in the span of \(k(\cdot,x_i)\). The orthogonal component satisfies \(f_\perp(x_i)=0\), so it changes no training prediction but increases the norm. Hence it cannot occur at an optimum.

Convexity of the loss is not required for this geometric statement. Convexity matters for optimization and uniqueness.

If exact interpolation is possible, the minimum-RKHS-norm interpolator satisfies

$$
f(\cdot)=\sum_i\alpha_i k(\cdot,x_i),
\qquad
K\alpha=y.
$$

When \(K\) is singular, the minimum-norm coefficient representative uses the pseudoinverse \(K^\dagger y\), while any null-space change produces the same function.

## Section 16 — Kernel ridge regression

Kernel ridge regression solves

$$
\min_{f\in\mathcal H}
\left\{
\frac1n\sum_{i=1}^n(y_i-f(x_i))^2
+\lambda\|f\|_{\mathcal H}^2
\right\}.
$$

By the representer theorem,

$$
\widehat f_\lambda(x)
=
\sum_i\alpha_i k(x,x_i),
$$

and a canonical coefficient vector is

$$
\boxed{
\alpha=(K+n\lambda I)^{-1}y.
}
$$

If \(K\) is singular, coefficient vectors may be nonunique, but the predicted function is unique for \(\lambda>0\). The displayed vector is a canonical representative.

Define

$$
q(x)=(k(x,x_1),\ldots,k(x,x_n))^\top.
$$

Then KRR is linear in the response vector:

$$
\widehat f_\lambda(x)
=q(x)^\top(K+n\lambda I)^{-1}y
=\sum_iw_i(x)y_i.
$$

At training points,

$$
\widehat y=H_\lambda y,
\qquad
H_\lambda=K(K+n\lambda I)^{-1}.
$$

Unlike local averaging, KRR weights may be negative and need not sum to one. Negative weights permit higher-order cancellation and are important for adaptation to smooth target functions.

## Section 17 — Statistical analysis of KRR and effective dimension

For random design, define the population covariance operator

$$
\Sigma=\mathbb E[\phi(X)\otimes\phi(X)].
$$

It links the RKHS geometry to prediction error:

$$
\|g\|_{L^2(p)}^2
=
\langle g,\Sigma g\rangle_{\mathcal H}.
$$

The population effective dimension is

$$
\boxed{
\mathcal N(\lambda)
=
\operatorname{tr}
\left[
\Sigma(\Sigma+\lambda I)^{-1}
\right].
}
$$

For a possibly misspecified target, define

$$
A(\lambda,f_*)
=
\inf_{f\in\mathcal H}
\left\{
\|f-f_*\|_{L^2(p)}^2+\lambda\|f\|_{\mathcal H}^2
\right\}.
$$

Ignoring technical multiplicative factors, KRR has the structure

$$
\mathbb E\|\widehat f_\lambda-f_*\|_{L^2(p)}^2
\lesssim
\frac{\sigma^2}{n}\mathcal N(\lambda)
+A(\lambda,f_*).
$$

If covariance eigenvalues satisfy

$$
\mu_j\asymp j^{-2s/d},
$$

then

$$
\mathcal N(\lambda)\asymp\lambda^{-d/(2s)}.
$$

If the target has Sobolev smoothness \(t\), the approximation term behaves like \(\lambda^{t/s}\). Balancing bias and variance yields

$$
\lambda_{\mathrm{opt}}
\asymp n^{-2s/(2t+d)},
\qquad
\boxed{
\operatorname{excess\ risk}
\asymp n^{-2t/(2t+d)}.
}
$$

For \(t=1\), this recovers the local-averaging rate \(n^{-2/(d+2)}\). Greater target smoothness gives faster rates, which is the smoothness adaptivity of KRR.

## Section 18 — Scalable kernel computation

An exact kernel matrix requires \(O(n^2)\) storage, and a direct KRR solve costs \(O(n^3)\).

Raw optimization in the coefficient vector \(\alpha\) can also be poorly conditioned because its Hessian contains factors of \(K\). If

$$
K=\Phi\Phi^\top,
$$

optimization in explicit coordinates has Hessian

$$
\frac1n\Phi^\top\Phi+\lambda I,
$$

whose smallest eigenvalue is at least \(\lambda\).

Nyström approximation selects \(m\) landmark columns \(I\):

$$
K
\approx
K(:,I)K(I,I)^{-1}K(I,:).
$$

This produces \(m\)-dimensional data-dependent features and typically costs \(O(nm^2)\).

Random features use a representation

$$
k(x,z)
=
\mathbb E_v[\varphi(x,v)\varphi(z,v)]
$$

and approximate it by

$$
\widetilde k(x,z)
=
\frac1m\sum_{j=1}^m
\varphi(x,v_j)\varphi(z,v_j).
$$

For translation-invariant kernels, random Fourier features use

$$
\sqrt2\cos(\omega^\top x+b),
$$

where \(\omega\) is sampled from the normalized spectral measure and \(b\) is uniform on \([0,2\pi]\).

Nyström is data-dependent; random features can be sampled before observing the training inputs.

## Section 19 — Ridgeless interpolation, implicit bias, and double descent

When \(d>n\) and \(XX^\top\) is invertible, the interpolation equations

$$
X\theta=y
$$

have infinitely many solutions. Gradient descent initialized at \(\theta_0=0\) remains in the row space of \(X\) and converges to

$$
\boxed{
\theta^\dagger
=X^\top(XX^\top)^{-1}y.
}
$$

This is the minimum-\(\ell_2\)-norm interpolator. In an RKHS, the analogous object is the minimum-RKHS-norm interpolating function.

For separable logistic regression, the parameter norm diverges, but the normalized direction converges to the hard-margin SVM direction. This is another implicit regularization effect.

Implicit bias is not automatically beneficial. It depends on parameterization, initialization, optimizer, and data structure. If the true solution is sparse, an implicit \(\ell_2\) preference may be inferior to explicit \(\ell_1\) regularization.

In the isotropic Gaussian linear model, the minimum-norm estimator has exact expected excess risk

$$
\mathbb E[R(\widehat\theta)-R^*]
=
\frac{\sigma^2d}{n-d-1},
\qquad d\le n-2,
$$

and

$$
\mathbb E[R(\widehat\theta)-R^*]
=
\frac{\sigma^2n}{d-n-1}
+\|\theta_*\|^2\frac{d-n}{d},
\qquad d\ge n+2.
$$

The variance explodes near the interpolation threshold \(d=n\), then decreases in the overparameterized regime. Properly tuned ridge regularization smooths or can remove this double-descent peak.

## Section 20 — KDE and local averaging

A smoothing kernel used in density estimation or local averaging is not the same mathematical object as an RKHS kernel.

For kernel density estimation in \(\mathbb R^d\),

$$
\widehat p_h(x)
=
\frac1{nh^d}
\sum_{i=1}^n
q\left(\frac{x-x_i}{h}\right),
$$

where \(q\ge0\) and usually \(\int q=1\). The factor \(h^{-d}\) is essential for density normalization.

The bandwidth controls a bias–variance tradeoff:

- small \(h\): local and low-bias, but noisy;
- large \(h\): stable, but oversmoothed.

For a twice-smooth one-dimensional density, the classical integrated squared-error balance is

$$
\operatorname{Variance}=O\left(\frac1{nh}\right),
\qquad
\operatorname{Bias}^2=O(h^4),
$$

which gives

$$
h_{\mathrm{opt}}\asymp n^{-1/5}.
$$

This rate should not be confused with Nadaraya–Watson regression under only a Lipschitz assumption.

## Section 21 — Nadaraya–Watson regression and Attention

Nadaraya–Watson regression uses normalized nonnegative weights:

$$
\widehat f_h(x)
=
\sum_iw_i(x)y_i,
\qquad
w_i(x)
=
\frac{q_h(x-x_i)}
{\sum_jq_h(x-x_j)}.
$$

Here \(q_h\) needs to be pointwise nonnegative, not positive semidefinite.

Conditioned on the inputs,

$$
\begin{aligned}
\mathbb E[
(\widehat f_h(x)-f_*(x))^2
\mid X_{1:n}
]
={}&
\left[
\sum_iw_i(x)(f_*(x_i)-f_*(x))
\right]^2\\
&+
\sum_iw_i(x)^2
\operatorname{Var}(Y_i\mid X_i).
\end{aligned}
$$

For a \(B\)-Lipschitz target in \(d\) dimensions,

$$
\operatorname{Bias}^2\asymp B^2h^2,
\qquad
\operatorname{Variance}\asymp\frac{\sigma^2}{nh^d}.
$$

Therefore

$$
\boxed{
h_{\mathrm{opt}}\asymp n^{-1/(d+2)},
\qquad
\operatorname{risk}\asymp n^{-2/(d+2)}.
}
$$

Consistency requires

$$
h\to0,
\qquad
nh^d\to\infty.
$$

With second-order smoothness and a suitable symmetric kernel, squared bias becomes \(O(h^4)\), leading to \(n^{-4/(d+4)}\).

Scaled dot-product Attention is

$$
\operatorname{Attention}(Q,K,V)
=
\operatorname{softmax}
\left(
\frac{QK^\top}{\sqrt d}
\right)V.
$$

It resembles Nadaraya–Watson because each query produces normalized nonnegative weights over values. The analogy is structural, not an equality of theories: query and key maps are learned, scores need not be symmetric or positive semidefinite, and Attention is not automatically an RKHS method.

## Section 22 — Unified comparison, workflow, and review questions

The three main weighted estimators are:

| Method | Similarity requirement | Weights | Main tuning parameter |
|---|---|---|---|
| Nadaraya–Watson | pointwise nonnegative window | nonnegative, sum to one | bandwidth \(h\) |
| Kernel ridge regression | positive-semidefinite Gram kernel | may be negative, need not sum to one | regularization \(\lambda\) |
| Attention | learned softmax scores | nonnegative, each row sums to one | learned \(Q,K,V\), temperature/scale |

A practical kernel-method workflow is:

1. Specify the prediction task and decide whether a linear model is plausible.
2. Standardize inputs and select a feature map or kernel family.
3. Check kernel validity and the meaning of its hyperparameters.
4. Tune kernel scale and regularization using validation or cross-validation.
5. Inspect conditioning, effective degrees of freedom, and support-vector count.
6. For large \(n\), use Nyström, random features, or iterative solvers.
7. Report whether the analysis is fixed design or random design.
8. Treat ridgeless interpolation as an algorithm-dependent minimum-norm choice, not as a generic property of all interpolators.

Review questions:

1. Why can ridge beat OLS without contradicting Gauss–Markov?
2. Derive both primal and dual ridge formulas.
3. Explain the difference between \(df_1(\lambda)\) and \(df_2(\lambda)\).
4. Why does finite-\(C\) soft SVM not necessarily equal hard SVM on separable data?
5. Prove that \(k(E,E')=|E\cap E'|\) is a valid kernel.
6. Prove the representer theorem using an orthogonal decomposition.
7. Explain why KRR weights may be negative while NW weights cannot.
8. State the conditions \(h\to0\) and \(nh^d\to\infty\) and interpret them.
9. Explain how gradient descent selects a minimum-norm interpolator.
10. Distinguish a smoothing kernel, a Mercer kernel, and an Attention score.